# Tech Challenge Fase 4
## 00 — Configuração independente do ambiente Databricks

### Objetivo
Validar o Volume existente, preparar os diretórios da Fase 4 e inventariar os arquivos de entrada.
Este notebook pode ser importado e executado em qualquer pasta de notebooks do Workspace,
inclusive antes de organizar o Git folder. Não importa módulos de src e não procura a raiz do Git.

### Pré-requisitos
Compute com Unity Catalog, Runtime 13.3 LTS ou superior (ou serverless compatível),
e permissões USE CATALOG, USE SCHEMA, READ VOLUME e WRITE VOLUME no Volume existente.
O notebook não cria Volume, External Location, bucket nem credenciais.


In [0]:
# Objetivo: definir os caminhos sem depender de arquivos externos.
# Justificativa: permitir executar o setup antes da organização do código no Git folder.
# Ação: centralizar o Volume, o prefixo da Fase 4 e os nove arquivos esperados.
import json
import platform
import importlib.metadata
from datetime import datetime, timezone

EXTERNAL_VOLUME = "workspace.default.vol_trio_drive"
VOLUME_ROOT = "/Volumes/workspace/default/vol_trio_drive"
PROJECT_PREFIX = "projetos/fiap/tech_challenge_fase4"
BASE_PATH = f"{VOLUME_ROOT}/{PROJECT_PREFIX}"
PROJECT_S3 = "s3://s3tc2/projetos/fiap/tech_challenge_fase4/"
DATA_PATH = f"{BASE_PATH}/data"
RAW_PATH = f"{DATA_PATH}/raw"
PROCESSED_PATH = f"{DATA_PATH}/processed"
REPORTS_PATH = f"{BASE_PATH}/reports"
CONFIG_PATH = f"{BASE_PATH}/config"
ARQUIVOS_RAW = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
print("Volume:", EXTERNAL_VOLUME)
print("Projeto no Databricks:", BASE_PATH)
print("Projeto no S3:", PROJECT_S3)


## Validação do Volume e de sua localização física
Apenas consultar o catálogo e listar o Volume não modifica dados. Antes de criar pastas,
confirmamos que o External Volume aponta para a raiz s3://s3tc2/, conforme o setup da Fase 3.
Se a localização divergir, a execução para antes de gravar: um prefixo diferente mudaria o destino físico.


In [0]:
# Objetivo: verificar o Volume existente antes de qualquer gravação.
# Justificativa: confirmar que os caminhos /Volumes correspondem ao destino S3 esperado.
# Ação: consultar os metadados e testar a listagem.
try:
    descricao_volume = spark.sql(f"DESCRIBE VOLUME {EXTERNAL_VOLUME}")
    linha_volume = descricao_volume.first()
    if linha_volume is None:
        raise ValueError("DESCRIBE VOLUME não retornou metadados.")
    # Algumas versões retornam storage_location/volume_type; outras, camelCase.
    metadados_originais = linha_volume.asDict()
    metadados_volume = {
        "".join(caractere for caractere in str(chave).lower() if caractere.isalnum()): valor
        for chave, valor in metadados_originais.items()
    }
    display(descricao_volume)
except Exception as exc:
    raise RuntimeError(
        "Não foi possível consultar workspace.default.vol_trio_drive. "
        "Confira o compute, o Volume e as permissões do Unity Catalog."
    ) from exc
campos_ausentes = {"storagelocation", "volumetype"} - set(metadados_volume)
if campos_ausentes:
    raise ValueError(
        "Não foi possível reconhecer as colunas de DESCRIBE VOLUME. "
        f"Colunas retornadas: {list(metadados_originais)}. "
        "Compartilhe os nomes das colunas exibidas para ajustar a leitura; nenhuma pasta foi criada."
    )
localizacao = str(metadados_volume.get("storagelocation") or "").strip().rstrip("/")
tipo_volume = str(metadados_volume.get("volumetype") or "").strip().upper()
if tipo_volume != "EXTERNAL" or localizacao != "s3://s3tc2":
    raise ValueError(
        f"Volume diferente do esperado: tipo={tipo_volume}, localização={localizacao}. "
        "Esperado: EXTERNAL sobre s3://s3tc2/. Confira DESCRIBE VOLUME antes de prosseguir."
    )
try:
    dbutils.fs.ls(VOLUME_ROOT)
except Exception as exc:
    raise RuntimeError("Volume não acessível. Confira READ VOLUME e o compute selecionado.") from exc
print("Volume e mapeamento S3 validados.")


In [0]:
# Objetivo: preparar os diretórios de dados da Fase 4.
# Justificativa: permitir os uploads antes da execução dos notebooks de processamento.
# Ação: criar somente as subpastas do projeto, de forma idempotente.
diretorios = [BASE_PATH, DATA_PATH, RAW_PATH, PROCESSED_PATH, REPORTS_PATH, CONFIG_PATH]
for caminho in diretorios:
    if not (caminho == BASE_PATH or caminho.startswith(BASE_PATH + "/")):
        raise ValueError(f"Caminho fora da Fase 4: {caminho}")
    dbutils.fs.mkdirs(caminho)
print("Diretórios criados ou já existentes:")
for caminho in diretorios:
    print(caminho)


## Inventário e orientação de upload
A ausência de arquivos é esperada no primeiro setup e não é tratada como falha.
O notebook termina com o estado AGUARDANDO_UPLOAD quando faltarem CSVs; depois do upload,
execute novamente para obter PRONTO_PARA_NOTEBOOK_01. Arquivos de tamanho zero também ficam pendentes.
Erros de permissão ou listagem continuam interrompendo a execução.

| Arquivos | Pasta dentro da Fase 4 |
| --- | --- |
| Nove CSVs originais do Kaggle | data/raw/ |
| JSONL e NPY existentes, opcionais para reaproveitamento | data/processed/ |


In [0]:
# Objetivo: conferir os arquivos sem exigir upload prévio ao setup.
# Justificativa: separar configuração concluída de dados disponíveis para processamento.
# Ação: inventariar os CSVs e verificar a presença dos artefatos opcionais.
itens_raw = {item.name: item for item in dbutils.fs.ls(RAW_PATH) if not item.isDir()}
inventario = [
    {"arquivo": nome, "existe": nome in itens_raw,
     "bytes": int(itens_raw[nome].size) if nome in itens_raw else 0}
    for nome in ARQUIVOS_RAW
]
display(spark.createDataFrame(inventario))
pendentes = [r["arquivo"] for r in inventario if not r["existe"] or r["bytes"] == 0]
itens_processed = {item.name for item in dbutils.fs.ls(PROCESSED_PATH) if not item.isDir()}
STATUS = "AGUARDANDO_UPLOAD" if pendentes else "PRONTO_PARA_NOTEBOOK_01"
print("Estado:", STATUS)
if pendentes:
    print("Envie ou substitua os arquivos vazios em:", RAW_PATH)
    print("Destino S3:", PROJECT_S3 + "data/raw/")
    for nome in pendentes:
        print("-", nome)
else:
    print("Os nove CSVs estão presentes e não vazios. Próxima etapa: Notebook 01.")
for nome in ["olist_reviews_documentos_rag.jsonl", "olist_reviews_embeddings.npy"]:
    print(nome, "presente" if nome in itens_processed else "não enviado (opcional no setup)")


In [0]:
# Objetivo: persistir configuração, inventário e versões da execução.
# Justificativa: registrar onde os dados serão lidos e o estado do preparo.
# Ação: gravar somente JSONs de controle, sem sobrescrever os dados de entrada.
registro_ambiente = {"python": platform.python_version()}
for nome in ["pandas", "numpy", "chromadb", "sentence-transformers", "torch", "transformers"]:
    try:
        registro_ambiente[nome] = importlib.metadata.version(nome)
    except importlib.metadata.PackageNotFoundError:
        registro_ambiente[nome] = "não instalado"
configuracao = {
    "project": "tech_challenge_fase4", "external_volume": EXTERNAL_VOLUME,
    "volume_root": VOLUME_ROOT, "s3_uri": PROJECT_S3, "data_root": BASE_PATH,
    "paths": {"raw": RAW_PATH, "processed": PROCESSED_PATH, "reports": REPORTS_PATH},
    "status": STATUS, "pendentes": pendentes,
    "execution_ts_utc": datetime.now(timezone.utc).isoformat(),
}
for caminho, objeto in [
    (f"{CONFIG_PATH}/config.json", configuracao),
    (f"{REPORTS_PATH}/00_ambiente.json", registro_ambiente),
    (f"{REPORTS_PATH}/00_inventario_raw.json", inventario),
]:
    dbutils.fs.put(caminho, json.dumps(objeto, ensure_ascii=False, indent=2), overwrite=True)
print("Setup concluído:", STATUS)
print("Configuração:", f"{CONFIG_PATH}/config.json")


## Organização do código e continuidade

O setup 00 está independente de src e do diretório onde o notebook está aberto no Workspace.
Ele não altera as pastas notebook, notebooks ou src que você já criou no Volume.
Essas pastas no catálogo armazenam arquivos no S3; sua existência não disponibiliza módulos Python
no Workspace nem inclui arquivos automaticamente nos commits.

Para executar os notebooks 01 a 03 do pacote anterior, mantenha no Git folder integrado do Workspace:

- notebooks/ com os arquivos ipynb;
- src/ ao lado de notebooks/, com __init__.py, config.py, artefatos.py e busca_vetorial.py.

Crie/importe essas pastas diretamente na interface do Databricks. Não é necessário criá-las primeiro
no site do GitHub. O conteúdo de src/config.py do pacote já usa o mesmo data_root deste setup.
Os notebooks 01 a 03 anteriores continuam dependendo desses módulos; esta correção remove a dependência
somente do setup 00. O config.json gravado no Volume registra os caminhos, mas não substitui src/config.py.

### Ordem de trabalho
1. Importe este notebook como notebook do Workspace e execute todas as células.
2. Faça upload dos CSVs em data/raw e, se desejar, do JSONL e NPY em data/processed, via Catalog/Volume ou S3.
3. Execute novamente o setup e confirme PRONTO_PARA_NOTEBOOK_01.
4. Organize notebooks/ e src/ no Git folder integrado e execute 01, 02 e 03 na ordem.
5. Faça commit e push da branch dev pelo Databricks para versionar o código.

O índice Chroma continua sendo reconstruído no disco temporário pelo Notebook 03; os artefatos persistem
no Volume. Nenhuma etapa do RAG ou de preparação textual foi alterada por esta correção do setup.
